## 双域合并 + 交叉过滤 + 数据集划分
加载电影和图书中间结果 → 合并 → 双域用户过滤 → 迭代式质量过滤 → 裁剪meta → 时序划分

In [1]:
import pandas as pd
import json, os, csv, sys

SAVE_DIR = "./processed/"
FINAL_DIR = "./final/"
os.makedirs(FINAL_DIR, exist_ok=True)

movie_df = pd.read_csv(f'{SAVE_DIR}/movie_interactions_filtered.csv')
book_df = pd.read_csv(f'{SAVE_DIR}/book_interactions_filtered.csv')

print(f'电影: {len(movie_df)} 交互, {movie_df["user_id"].nunique()} 用户, {movie_df["item_id"].nunique()} 物品')
print(f'图书: {len(book_df)} 交互, {book_df["user_id"].nunique()} 用户, {book_df["item_id"].nunique()} 物品')

df = pd.concat([movie_df, book_df], ignore_index=True)
print(f'\n合并: {len(df)} 交互, {df["user_id"].nunique()} 用户, {df["item_id"].nunique()} 物品')

电影: 3114700 交互, 238403 用户, 106386 物品
图书: 8145379 交互, 678456 用户, 429001 物品

合并: 11260079 交互, 850402 用户, 535387 物品


In [2]:
# ===== 双域交叉过滤 =====

# 1. 仅保留双域用户
user_domain_count = df.groupby('user_id')['domain'].nunique()
dual_users = user_domain_count[user_domain_count == 2].index
df = df[df['user_id'].isin(dual_users)]
print(f'双域用户: {len(df)} 交互, {df["user_id"].nunique()} 用户')

双域用户: 2466486 交互, 66457 用户


In [3]:
# 2. 每域至少 >=3 条交互
user_domain_cnt = df.groupby(['user_id', 'domain']).size().unstack(fill_value=0)
valid_users = user_domain_cnt[
    (user_domain_cnt['movie'] >= 3) &
    (user_domain_cnt['book'] >= 3)
].index
df = df[df['user_id'].isin(valid_users)]
print(f'每域>=3: {len(df)} 交互, {df["user_id"].nunique()} 用户')

每域>=3: 2466486 交互, 66457 用户


In [4]:
# 3. 用户总交互 >= 10
user_total = df.groupby('user_id').size()
df = df[df['user_id'].isin(user_total[user_total >= 10].index)]
print(f'用户>=10: {len(df)} 交互, {df["user_id"].nunique()} 用户, {df["item_id"].nunique()} 物品')

用户>=10: 2466486 交互, 66457 用户, 426248 物品


In [7]:
# 4. 迭代式双向过滤（含双域约束）
MIN_ITEM = 7
MIN_USER = 7

while True:
    n_u = df['user_id'].nunique()
    n_i = df['item_id'].nunique()
    
    item_cnt = df.groupby('item_id').size()
    df = df[df['item_id'].isin(item_cnt[item_cnt >= MIN_ITEM].index)]
    
    user_cnt = df.groupby('user_id').size()
    df = df[df['user_id'].isin(user_cnt[user_cnt >= MIN_USER].index)]
    
    # 保持双域条件
    udc = df.groupby('user_id')['domain'].nunique()
    df = df[df['user_id'].isin(udc[udc == 2].index)]
    udc2 = df.groupby(['user_id', 'domain']).size().unstack(fill_value=0)
    valid = udc2[(udc2['movie'] >= 5) & (udc2['book'] >= 5)].index
    df = df[df['user_id'].isin(valid)]
    
    if df['user_id'].nunique() == n_u and df['item_id'].nunique() == n_i:
        break
    print(f"  迭代中... 用户={df['user_id'].nunique()}, 物品={df['item_id'].nunique()}, 交互={len(df)}")

print(f"\n===== 最终结果 =====")
print(f"交互: {len(df)}")
print(f"用户: {df['user_id'].nunique()}")
print(f"物品: {df['item_id'].nunique()}")
print(f"电影域: {len(df[df['domain']=='movie'])} 交互, {df[df['domain']=='movie']['item_id'].nunique()} 物品")
print(f"图书域: {len(df[df['domain']=='book'])} 交互, {df[df['domain']=='book']['item_id'].nunique()} 物品")

  迭代中... 用户=31120, 物品=81236, 交互=1071883
  迭代中... 用户=24716, 物品=57393, 交互=845639
  迭代中... 用户=22204, 物品=49549, 交互=760098
  迭代中... 用户=21071, 物品=46295, 交互=722492
  迭代中... 用户=20546, 物品=44886, 交互=704634
  迭代中... 用户=20277, 物品=44164, 交互=695819
  迭代中... 用户=20144, 物品=43810, 交互=691682
  迭代中... 用户=20084, 物品=43655, 交互=689895
  迭代中... 用户=20058, 物品=43598, 交互=689003
  迭代中... 用户=20041, 物品=43557, 交互=688347
  迭代中... 用户=20034, 物品=43542, 交互=688162
  迭代中... 用户=20030, 物品=43529, 交互=688016
  迭代中... 用户=20030, 物品=43528, 交互=688010

===== 最终结果 =====
交互: 688010
用户: 20030
物品: 43528
电影域: 394876 交互, 21280 物品
图书域: 293134 交互, 22248 物品


In [8]:
# ===== 构建ID映射 =====
df = df.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)

user2id = {uid: idx for idx, uid in enumerate(df['user_id'].unique())}
item2id = {iid: idx for idx, iid in enumerate(df['item_id'].unique())}
domain2id = {'movie': 0, 'book': 1}

df['user_id'] = df['user_id'].map(user2id)
df['item_id'] = df['item_id'].map(item2id)
df['domain'] = df['domain'].map(domain2id)

print(f'用户ID范围: 0~{len(user2id)-1}')
print(f'物品ID范围: 0~{len(item2id)-1}')

用户ID范围: 0~20029
物品ID范围: 0~43527


In [9]:
# ===== 时序划分：每个用户最后2条作为 val/test =====
df['reverse_seq_pos'] = df.groupby('user_id').cumcount(ascending=False)

train_df = df[df['reverse_seq_pos'] >= 2].drop(columns=['reverse_seq_pos'])
val_df   = df[df['reverse_seq_pos'] == 1].drop(columns=['reverse_seq_pos'])
test_df  = df[df['reverse_seq_pos'] == 0].drop(columns=['reverse_seq_pos'])

print(f'训练集: {len(train_df)} | 验证集: {len(val_df)} | 测试集: {len(test_df)}')

train_df.to_csv(f'{FINAL_DIR}/train.csv', index=False)
val_df.to_csv(f'{FINAL_DIR}/val.csv', index=False)
test_df.to_csv(f'{FINAL_DIR}/test.csv', index=False)

with open(f'{FINAL_DIR}/user2id.json', 'w', encoding='utf-8') as f:
    json.dump(user2id, f, ensure_ascii=False, indent=2)
with open(f'{FINAL_DIR}/item2id.json', 'w', encoding='utf-8') as f:
    json.dump(item2id, f, ensure_ascii=False, indent=2)
with open(f'{FINAL_DIR}/domain2id.json', 'w', encoding='utf-8') as f:
    json.dump(domain2id, f, ensure_ascii=False, indent=2)

print(f'\n数据已保存至 {FINAL_DIR}/')

训练集: 647950 | 验证集: 20030 | 测试集: 20030

数据已保存至 ./final//


In [15]:
# ===== 裁剪meta：只保留最终物品集中的物品 =====
import sys
import csv
max_int = sys.maxsize
while True:
    try:
        csv.field_size_limit(max_int)
        break
    except OverflowError:
        max_int = max_int // 10

id2item = {v: k for k, v in item2id.items()}
final_items = set(id2item.values())
print(f'最终有效物品: {len(final_items)}')

for domain, src, dst in [
    ('movie', f'{SAVE_DIR}/movie_meta_clean.csv', f'{FINAL_DIR}/item_meta_movie.csv'),
    ('book', f'{SAVE_DIR}/book_meta_clean.csv', f'{FINAL_DIR}/item_meta_book.csv'),
]:
    kept = 0
    with open(src, encoding='utf-8') as f_in, \
         open(dst, 'w', encoding='utf-8', newline='') as f_out:
        reader = csv.DictReader(f_in)
        writer = csv.DictWriter(f_out, fieldnames=reader.fieldnames)
        writer.writeheader()
        for row in reader:
            if row['parent_asin'] in final_items:
                row['parent_asin'] = str(item2id[row['parent_asin']])
                writer.writerow(row)
                kept += 1
    print(f'{domain}: 保留 {kept} 条')

print(f'\nmeta已保存至 {FINAL_DIR}/item_meta_*.csv')

最终有效物品: 43528
movie: 保留 21280 条
book: 保留 22248 条

meta已保存至 ./final//item_meta_*.csv


In [18]:
# ===== 合并meta供URL下载图片用 =====
movie_meta = pd.read_csv(f'{FINAL_DIR}/item_meta_movie.csv')
book_meta = pd.read_csv(f'{FINAL_DIR}/item_meta_book.csv')
movie_meta['domain'] = 0
book_meta['domain'] = 1
merged_meta = pd.concat([movie_meta, book_meta], ignore_index=True)
merged_meta['parent_asin'] = merged_meta['parent_asin'].astype(int)
merged_meta = merged_meta.sort_values(by='parent_asin').reset_index(drop=True)
merged_meta.to_csv(f'{FINAL_DIR}/item_meta_merged.csv', index=False)

print(f'电影: {len(movie_meta)} | 图书: {len(book_meta)} | 合并: {len(merged_meta)}')
print(f'列: {list(merged_meta.columns)}')
print(f'已保存: {FINAL_DIR}/item_meta_merged.csv')

电影: 21280 | 图书: 22248 | 合并: 43528
列: ['parent_asin', 'title', 'description', 'imUrl', 'domain']
已保存: ./final//item_meta_merged.csv
